# cuconv on a Colab T4

Runs the whole repo end to end: build the kernels, check every stage against cuDNN, benchmark, profile, draw the chart and the feature-map grid. Runtime > Change runtime type > T4 GPU, then Runtime > Run all. Takes about ten minutes.

In [ ]:
!nvidia-smi
!nvcc --version | tail -2
import torch; print('torch', torch.__version__, 'cuda', torch.version.cuda, 'cudnn', torch.backends.cudnn.version())

In [ ]:
import os
if not os.path.exists('cuconv'):
    !git clone --quiet https://github.com/noorps/cuconv.git
%cd cuconv
!git pull --quiet && git log --oneline -1

In [ ]:
import time
LABEL = time.strftime('t4-%Y%m%d-%H%M')
print(LABEL)
!bash run.sh $LABEL 2>&1 | tee results/$LABEL/run.log

In [ ]:
from IPython.display import Image, display
display(Image(f'results/{LABEL}/gflops.png', width=1100))
display(Image(f'results/{LABEL}/featuremaps.png', width=1100))

In [ ]:
import json
b = json.load(open(f'results/{LABEL}/bench.json'))
print(json.dumps(b['device'], indent=1))
for r in b['rows']:
    print(f"{r['shape']:<26}{r['stage']:<10}{r['gflops']:>8.1f} GFLOP/s  [{r['gflops_min']:.1f}-{r['gflops_max']:.1f}]  {r['pct_cudnn']:6.1f}% of cuDNN")

In [ ]:
!zip -qr /content/{LABEL}.zip results/{LABEL}
from google.colab import files
files.download(f'/content/{LABEL}.zip')